# RAG on a Book: Build From Scratch

In this assignment you will build a **complete RAG system from scratch** so that a **language model of your choice** can answer questions about a book. You will:

1. **Load the book** (PDF or plain text)
2. **Chunk the text** into pieces suitable for retrieval
3. **Embed chunks** using a sentence-transformers model
4. **Retrieve** relevant chunks for each question
5. **Generate answers** with your chosen chat model using the retrieved context

## Can the book be a PDF? Will chunking be easy?

- **Yes, the book can be a PDF.** We use **PyMuPDF** to extract text. It works well for normal (non-scanned) PDFs. Scanned PDFs (images of pages) need OCR and are harder.
- **Chunking is straightforward** once you have raw text: split by paragraphs, or by a fixed size (e.g. number of characters or sentences) with optional overlap. You will implement a simple chunking strategy.

## What you need

- The book for this assignment is **WMD.pdf**, in this folder.
- **Ollama** installed; pull the chat model you plan to use (see "Choose chat model" below), e.g. `ollama pull gemma3:270m`. For **gpt-oss:120b-cloud**, set `OLLAMA_HOST` if using a remote server.
- Python 3.8+

## Part 1: Setup and installation

Install the required libraries. You need: sentence-transformers (embeddings), ollama (chat), numpy, scikit-learn (similarity), and **PyMuPDF** for PDF text extraction.

In [ ]:
# Run this cell once
%pip install sentence-transformers ollama numpy scikit-learn pymupdf

## Part 2: Imports

Import everything you will need for loading the book, chunking, embedding, retrieval, and calling the LLM.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import ollama
import re
from pathlib import Path
from typing import List, Tuple

# PyMuPDF is imported as fitz
import fitz

## Part 2b: Choose chat model

Set **CHAT_MODEL** to one of: **gpt-oss:120b-cloud**, **gemma3:270m**, **gemma3:27B**, **qwen3:4b**. The RAG and no-RAG pipelines will use this model.

- **gpt-oss:120b-cloud** — Cloud-hosted (set `OLLAMA_HOST` if using a remote Ollama server)
- **gemma3:270m** — Small, fast local model
- **gemma3:27B** — Larger local model (more RAM)
- **qwen3:4b** — 4B-parameter local model

In [ ]:
# Choose one of: 'gpt-oss:120b-cloud', 'gemma3:270m', 'gemma3:27B', 'qwen3:4b'
CHAT_MODEL = 'gemma3:270m'

print(f"Using chat model: {CHAT_MODEL}")

## Part 3: Load the book (PDF or text)

Implement a function that:
- If the file is a **.pdf**, use PyMuPDF to extract text from every page and concatenate it.
- If the file is a **.txt**, read it with standard file I/O.

Return a single string containing the full book text. Handle encoding (e.g. UTF-8) and strip extra whitespace where appropriate.

In [ ]:
def load_book(file_path: str) -> str:
    """
    Load a book from a PDF or .txt file.
    
    Args:
        file_path: Path to the PDF or .txt file.
    
    Returns:
        Full book text as a single string.
    """
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    suffix = path.suffix.lower()
    
    if suffix == ".pdf":
        # Use PyMuPDF to extract text from each page
        doc = fitz.open(file_path)
        parts = []
        for page in doc:
            parts.append(page.get_text())
        doc.close()
        text = "\n".join(parts)
    elif suffix == ".txt":
        with open(file_path, "r", encoding="utf-8", errors="replace") as f:
            text = f.read()
    else:
        raise ValueError(f"Unsupported format: {suffix}. Use .pdf or .txt")
    
    # Normalize whitespace: collapse multiple newlines to double newline (paragraph break)
    text = re.sub(r"\n{3,}", "\n\n", text.strip())
    return text

### Test loading

Set `BOOK_PATH` to your PDF or .txt file and run the cell. Print the length of the text and a short preview.

In [ ]:
# Book file: WMD.pdf is in the same folder as this notebook
BOOK_PATH = "WMD.pdf"


full_text = load_book(BOOK_PATH)
print(f"Loaded {len(full_text)} characters (~{len(full_text.split())} words)")
print("Preview (first 500 chars):")
print(full_text[:500])

## Part 4: Chunk the text

Chunking turns the long book string into a list of shorter strings (chunks) that you will embed and search. Two simple strategies:

1. **By paragraphs**: split on double newline `\n\n`, then optionally merge very short paragraphs so chunks are not too small.
2. **By fixed size**: split every N characters (or every N sentences) with a small overlap to avoid cutting mid-sentence.

Implement a function `chunk_text(text, strategy='paragraphs', max_chunk_size=500, overlap=50)` that returns a list of chunk strings. Use either strategy (or both and compare). Keep chunks small enough that several fit in your chat model's context (e.g. **32K tokens** for gemma3, ~24K words) along with the question and your instructions.

In [ ]:
def chunk_text(
    text: str,
    strategy: str = "paragraphs",
    max_chunk_size: int = 500,
    overlap: int = 50,
) -> list[str]:
    """
    Split the book text into chunks.
    
    Args:
        text: Full book text.
        strategy: 'paragraphs' (split on double newline) or 'fixed' (by character count with overlap).
        max_chunk_size: For 'fixed', target size in characters. For 'paragraphs', max chars per chunk when merging.
        overlap: For 'fixed', number of overlapping characters between consecutive chunks.
    
    Returns:
        List of chunk strings.
    """
    if strategy == "paragraphs":
        raw_paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        chunks = []
        current = []
        current_len = 0
        for p in raw_paragraphs:
            if current_len + len(p) + 2 > max_chunk_size and current:
                chunks.append("\n\n".join(current))
                current = []
                current_len = 0
            current.append(p)
            current_len += len(p) + 2
        if current:
            chunks.append("\n\n".join(current))
        return chunks
    
    elif strategy == "fixed":
        chunks = []
        start = 0
        while start < len(text):
            end = start + max_chunk_size
            chunk = text[start:end]
            if not chunk.strip():
                start = end - overlap
                continue
            chunks.append(chunk.strip())
            start = end - overlap
        return [c for c in chunks if c]
    
    else:
        raise ValueError("strategy must be 'paragraphs' or 'fixed'")


# Chunk the book
chunks = chunk_text(full_text, strategy="paragraphs", max_chunk_size=500)
print(f"Created {len(chunks)} chunks")
print("First chunk preview:")
print(chunks[0][:400] if chunks else "(none)")

## Part 5: Embedding model and chunk embeddings

Load the same embedding model as in the demo (**all-MiniLM-L6-v2**) and encode all chunks. Store the result in a matrix (e.g. numpy array) of shape `(num_chunks, embedding_dim)`.

In [ ]:
print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks, convert_to_tensor=False)
print(f"Shape of chunk_embeddings: {chunk_embeddings.shape}")

## Part 6: Retrieval function

Implement a function `retrieve(query, chunk_embeddings, chunks, top_k=3)` that:
1. Encodes the query with the same embedding model.
2. Computes cosine similarity between the query embedding and all chunk embeddings.
3. Returns the top-k chunks (and optionally their similarity scores).

In [ ]:
def retrieve(
    query: str,
    chunk_embeddings: np.ndarray,
    chunks: list[str],
    top_k: int = 3,
) -> list[tuple[str, float]]:
    """
    Return the top-k chunks most similar to the query.
    
    Returns:
        List of (chunk_text, similarity_score) for the top-k chunks.
    """
    query_embedding = embedding_model.encode([query], convert_to_tensor=False)
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    return [(chunks[i], float(similarities[i])) for i in top_indices]

## Part 7: RAG — ask the book a question

Combine retrieval and generation:
1. Take a user question.
2. Retrieve the top-k relevant chunks.
3. Build a prompt that includes the retrieved context and the question, and instructs the model to answer only from the context.
4. Call the selected **chat model** (CHAT_MODEL) via Ollama with this prompt and print the answer.

Remember: many models have a large context (e.g. **32K tokens** for gemma3). Keep the total prompt (instructions + context + question) within the model's limit.

In [ ]:
def answer_question(
    question: str,
    chunk_embeddings: np.ndarray,
    chunks: list[str],
    top_k: int = 3,
) -> str:
    """
    Use RAG to answer a question about the book.
    """
    retrieved = retrieve(question, chunk_embeddings, chunks, top_k=top_k)
    context_parts = [chunk for chunk, _ in retrieved]
    context_text = "\n\n---\n\n".join(context_parts)
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the following text from a book. If the text does not contain enough information, say so.

Text from the book:
{context_text}

Question: {question}

Answer:"""
    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]


# Example question
question = "What is the main theme of the book?"
answer = answer_question(question, chunk_embeddings, chunks, top_k=3)
print(f"Question: {question}")
print(f"Answer: {answer}")

## Part 8: RAG vs No-RAG comparison (10 test questions)

Run the same questions **with RAG** (retrieved book context) and **without RAG** (selected model alone). Compare the answers to see when RAG improves performance.

**10 test questions:**
1. What is the main theme or argument of the book?
2. Who are the key figures or characters mentioned?
3. What evidence or examples does the book give to support its main points?
4. How does the book define its most important terms or concepts?
5. What events or developments does the book describe?
6. What conclusions or recommendations does the book reach?
7. What risks, challenges, or criticisms does the book discuss?
8. When or where is the setting (if applicable)?
9. How does the book compare or contrast different ideas or positions?
10. What does the book say about [choose a specific topic from the book]?

In [ ]:
def answer_without_rag(question: str) -> str:
    """Ask the selected chat model the question with no book context (no RAG)."""
    prompt = f"""You are a helpful assistant. The following question is about the book *Weapons of Math Destruction* by Cathy O'Neil. Answer based on your general knowledge of this book. If you do not know, say so.

Question: {question}

Answer:"""
    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]


TEST_QUESTIONS = [
    "What is the main theme or argument of the book?",
    "Who are the key figures or characters mentioned?",
    "What evidence or examples does the book give to support its main points?",
    "How does the book define its most important terms or concepts?",
    "What events or developments does the book describe?",
    "What conclusions or recommendations does the book reach?",
    "What risks, challenges, or criticisms does the book discuss?",
    "When or where is the setting (if applicable)?",
    "How does the book compare or contrast different ideas or positions?",
    "What does the book say about the main topic or subject?",
]

print("Defined answer_without_rag(question) and TEST_QUESTIONS (10 questions).")
print("Run the next cell to compare RAG vs No-RAG for each question.")

In [ ]:
for i, q in enumerate(TEST_QUESTIONS, 1):
    print("=" * 70)
    print(f"Question {i}: {q}")
    print("=" * 70)
    print("\n--- With RAG (book context) ---")
    ans_rag = answer_question(q, chunk_embeddings, chunks, top_k=3)
    print(ans_rag)
    print(f"\n--- Without RAG ({CHAT_MODEL} only) ---")
    ans_no_rag = answer_without_rag(q)
    print(ans_no_rag)
    print()

### Compare and reflect

For each question, note:
- Did **RAG** give a more accurate or specific answer (grounded in the book)?
- Did **no RAG** hallucinate or give a generic answer?
- For which questions did RAG help most? When did it matter less?

## Part 9: Try your own questions

Ask several questions about your book. Observe how retrieval quality (choice of chunks) affects the answer. Experiment with `top_k` and, if you have time, with different chunking strategies (paragraph vs fixed size, chunk size, overlap).

In [ ]:
questions = [
    "Who is the main character?",
    "What happens at the end?",
]

for q in questions:
    a = answer_question(q, chunk_embeddings, chunks, top_k=3)
    print(f"Q: {q}")
    print(f"A: {a}")
    print("-" * 60)

## Part 10: Assignment tasks

1. **RAG vs No-RAG**: Run the 10 test questions in Part 8. For each, compare the answer with RAG vs without RAG. Summarize: when did RAG clearly help? When did the model without RAG hallucinate or give a generic answer?
2. **Load the book**: Use WMD.pdf (or another PDF/.txt). Report any issues with PDF extraction (e.g. odd order, missing pages).
3. **Chunking**: Try both paragraph-based and fixed-size chunking. Compare retrieval or answer quality for a few questions.
4. **Context length**: Estimate how many tokens your typical prompt uses (instruction + context + question). Ensure you stay within your chosen model's context limit (e.g. 32K for gemma3).
5. **top_k**: Experiment with top_k = 1, 2, 3, 5. How does it change answers?
6. **Reflection**: What would you do to improve this RAG system (e.g. better chunking, reranking, or a different model)?